# 02 — Classer une image : une proposition, pas une certitude

Nous savons maintenant ouvrir une photo et préparer ses pixels avec OpenCV.
Nous allons lui poser une première question : **« Quelle catégorie décrit le mieux cette image ? »**

Regardons notre photo de rue. Un bus attire l'œil, mais des personnes et des bâtiments
sont également présents. Le modèle va pourtant proposer **une étiquette pour l'image entière**.
Nous allons obtenir cette proposition, lire son score, puis décider ce que l'on peut en faire.

<img src="../donnees/images/bus.jpg" alt="Photo de rue utilisée dans les exemples : un bus, plusieurs personnes et des bâtiments." width="340">

Notre chemin : **poser la question → charger le modèle → calculer les scores → les interpréter → décider → voir la limite**.

Photo : [exemple officiel Ultralytics](https://github.com/ultralytics/ultralytics/blob/e544db43838fd058370cad216bf82a7639f29b35/ultralytics/assets/bus.jpg), fournie dans `donnees/images`.

Les **figures de référence** sont enregistrées dans ce cours : elles restent
visibles sans exécuter les cellules. Le code produit vos propres résultats ;
si vous changez une image ou un réglage, ils peuvent différer de ces références.


## 1. Une classe est simplement une catégorie

**Classer une image**, c'est lui associer une catégorie, aussi appelée une **classe**.
Son nom est l'**étiquette** : par exemple « autobus ».

Un **score** est ici un nombre entre 0 et 1 qui représente la préférence du
modèle pour une catégorie. Le modèle donne un score à chacune de ses catégories connues. La catégorie
avec le plus grand score constitue sa première proposition. C'est une **classification globale** :
une réponse sur l'image, pas une réponse pour chaque objet visible.

![Une photo produit un classement de catégories, puis une étiquette globale, sans position ni comptage.](../ressources/illustrations/02_question_classification.svg)

Il ne peut donc pas répondre à « Combien de personnes sont là ? » ni « Où est le bus ? ».
Gardons ces questions : elles nous conduiront naturellement à la détection.


## 2. Utiliser un modèle qui a déjà appris

Nous utilisons **YOLO11n-cls**, un petit réseau de neurones déjà entraîné.
`n` désigne la variante « nano », et `cls` la tâche de classification.
Le nom YOLO apparaît aussi en détection, mais **ce fichier-ci classe les images**.

L'entraînement a ajusté les **poids**, les nombres internes qui déterminent ses calculs,
à partir d'images déjà étiquetées. L'**inférence** consiste à utiliser ces poids pour
calculer une réponse sur une nouvelle image. Nous n'entraînons rien ici.

![Différence entre l’apprentissage déjà réalisé et l’inférence réalisée dans ce notebook.](../ressources/illustrations/02_apprentissage_inference.svg)

Ce modèle connaît les **1 000 catégories d'ImageNet-1k**, un ensemble d'images de référence.
Il ne peut pas inventer une nouvelle catégorie absente de cette liste.
Les noms sont en anglais et parfois très précis : nous conserverons les noms réellement renvoyés.

Sources : [modèles YOLO11](https://docs.ultralytics.com/models/yolo11/)
et [ImageNet-1k](https://docs.ultralytics.com/datasets/classify/imagenet/).


## 3. Retrouver notre photo dans le projet

Commençons par la photo, avant le modèle. `Path` représente un chemin de fichier.
La variable `ROOT` retrouvera le dossier du projet même si le notebook est ouvert depuis
`notebooks`. Les caches, petits fichiers techniques créés par les bibliothèques,
resteront dans `travaux`. Ce premier bloc prépare les chemins et les caches ;
les opérations de vision commencent ensuite. Il reprend la recherche de dossier
du notebook 01 : `next(...)` choisit le premier dossier contenant nos ressources.

OpenCV lira les pixels en **BGR** : bleu, vert, rouge. Matplotlib attend du **RGB** :
rouge, vert, bleu. La conversion servira uniquement à l'affichage.


In [ ]:
from pathlib import Path
import os

# Chercher le dossier qui contient les données et les modèles du cours.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "modeles").is_dir() and (p / "donnees").is_dir())

# Garder les fichiers techniques dans le projet, sans synchronisation distante.
config_dir = ROOT / "travaux" / "config"
config_dir.mkdir(parents=True, exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(config_dir)
os.environ["YOLO_AUTOINSTALL"] = "false"  # Ne rien installer depuis le notebook.
os.environ["YOLO_OFFLINE"] = "true"
os.environ["MPLCONFIGDIR"] = str(config_dir / "matplotlib")

import cv2
import matplotlib.pyplot as plt


In [ ]:
image_path = ROOT / "donnees" / "images" / "bus.jpg"
image_bgr = cv2.imread(str(image_path))  # Charger la photo en mémoire.
if image_bgr is None:
    raise FileNotFoundError(f"Photo introuvable ou illisible : {image_path}")

# Conserver la version BGR ; créer une version RGB pour l'affichage seulement.
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(5, 6))
plt.imshow(image_rgb)
plt.title("La photo complète : quelle étiquette proposer ?")
plt.axis("off")
plt.show()
print("Dimensions (hauteur, largeur, canaux) :", image_bgr.shape)


Quelle catégorie proposeriez-vous pour la photo entière, avant de regarder le résultat du modèle ?


## 4. Charger les calculs appris

Le fichier `.pt` est un fichier de modèle PyTorch, la bibliothèque de calcul utilisée ici.
Le fichier fourni contient les poids et les informations nécessaires pour reconstruire le réseau.
`YOLO(...)` charge ce modèle ; cela ne relance pas son entraînement.

Nous vérifions d'abord sa présence afin d'éviter tout téléchargement automatique.
Utiliser uniquement le fichier officiel fourni : un fichier de modèle inconnu n'est pas anodin.


In [ ]:
poids = ROOT / "modeles" / "yolo11n-cls.pt"
if not poids.is_file():
    raise FileNotFoundError("Modèle absent : suivre INSTALLATION.txt avant de continuer.")

from ultralytics import YOLO, settings
settings.update({"sync": False})  # Désactiver la synchronisation facultative.

classifieur = YOLO(str(poids))  # Les poids sont lus depuis le disque local.
assert classifieur.task == "classify", "Ce notebook attend un modèle de classification."
print("Tâche :", classifieur.task)
print("Catégories disponibles :", len(classifieur.names))


## 5. Respecter la préparation attendue par le modèle

Le **prétraitement** prépare les pixels pour les calculs du réseau : taille,
couleurs et échelle des nombres. Ici, `predict` prend notre image OpenCV en BGR
et effectue déjà les transformations attendues.

Avec **nos poids de classification** et `imgsz=224`, il réduit d'abord la photo
en conservant ses proportions, puis garde le **carré central de 224 × 224 pixels**.
On perd donc du contenu en haut et en bas : réduire et recadrer sont bien deux gestes.

![La préparation d'une photo verticale vers un carré peut retirer du contenu.](../ressources/illustrations/02_pretraitement.svg)

Les couleurs sont rangées en RGB pour le réseau. Les valeurs de `0` à `255`
deviennent des nombres de `0` à `1` : par exemple `128 / 255 ≈ 0,502`.
Le tableau est aussi réorganisé pour les calculs du modèle. Ce n'est pas un
nouvel apprentissage : les pixels sont seulement présentés dans le format attendu.

**Ne refaisons pas ces opérations avant l'appel.** Fournissons la photo BGR
à `predict`, qui applique lui-même sa recette. Une autre famille de modèles
pourrait attendre une préparation différente.
`device="cpu"` utilise le processeur, sans carte graphique dédiée.

Source : [classification et transformations d'image](https://docs.ultralytics.com/tasks/classify/).


**Figure de référence · visible sans exécution**

![Photo de rue originale à gauche et carré de 224 pixels réellement préparé pour le classifieur à droite.](../ressources/illustrations/02_preparation_reelle.png)

Cette image a été produite avec les transformations du modèle fourni, pas avec un recadrage dessiné à la main. Repérez ce qui disparaît en haut et en bas : le modèle ne reçoit pas tout le cadrage original.


In [ ]:
resultats = classifieur.predict(
    source=image_bgr,  # Une image OpenCV : canaux BGR, pas la version RGB d'affichage.
    imgsz=224,        # Taille de l'entrée préparée pour ce classifieur.
    device="cpu",    # Faire les calculs sur le processeur de l'ordinateur.
    verbose=False,   # Éviter les messages techniques à chaque prédiction.
    save=False,      # Ne pas créer automatiquement de dossiers de résultats.
)
resultat = resultats[0]  # Une seule image en entrée : son résultat est le premier.
assert resultat.probs is not None, "Le résultat doit contenir des scores de classification."


## 6. Lire la proposition sans lui faire dire davantage

`resultat.probs` contient les scores des catégories. **Top-1** signifie « premier du classement ».

Trois informations sont distinctes :

| Élément | Signification |
|---|---|
| `top1` | Le numéro de la catégorie arrivée en tête. |
| `names[numero]` | Le nom associé à ce numéro. |
| `top1conf` | Le score attribué à cette catégorie, entre 0 et 1. |

`float(...)` transforme le nombre renvoyé par la bibliothèque de calcul en nombre Python.
Le format `:.1%` l'affiche en pourcentage, avec un chiffre après la virgule.

Source : [résultats de classification et scores](https://docs.ultralytics.com/reference/engine/results/).


In [ ]:
numero = resultat.probs.top1
etiquette = resultat.names[numero]
score = float(resultat.probs.top1conf)

print("Numéro interne :", numero)
print("Étiquette proposée :", etiquette)
print(f"Score du modèle : {score:.1%}")  # Un affichage, pas un taux de réussite mesuré.


## 7. Voir les autres propositions

Un seul score cache les alternatives. Le **top-5** contient les cinq catégories les mieux classées,
pas cinq objets détectés. Nous allons représenter leurs scores avec des barres.

`top5` fournit leurs numéros. Nous utilisons ensuite chaque numéro pour retrouver le nom
et le score correspondant. Les **1 000 scores** se partagent une somme proche de 1 ;
les cinq premiers ne représentent donc pas forcément toute cette somme.


In [ ]:
numeros_top5 = resultat.probs.top5
noms_top5 = []
scores_top5 = []

for numero_classe in numeros_top5:
    noms_top5.append(resultat.names[numero_classe])  # Retrouver le nom de la catégorie.
    scores_top5.append(float(resultat.probs.data[numero_classe]))  # Retrouver son score.

for nom, valeur in zip(noms_top5, scores_top5):  # zip associe chaque nom à son score.
    print(f"{nom:25s} {valeur:.1%}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
barres = ax.barh(noms_top5, scores_top5, color="#4474a9")  # Une barre par catégorie.
ax.invert_yaxis()  # Placer la meilleure proposition en haut.
ax.set_xlim(0, 1)  # Garder la même échelle de score, de 0 à 1.
ax.set_xlabel("Score du modèle")
ax.set_title("Les cinq meilleures propositions pour cette photo")
for barre, valeur in zip(barres, scores_top5):
    ax.text(min(valeur + 0.015, 0.86), barre.get_y() + barre.get_height() / 2,
            f"{valeur:.1%}", va="center")
plt.tight_layout()
plt.show()


**Figure de référence · visible sans exécution**

![Cinq catégories les mieux notées par le classifieur sur la photo du bus.](../ressources/illustrations/02_scores_top5.png)

Sur le calcul de référence, `minibus` arrive en tête, puis `police_van`. Ces noms sont des catégories du modèle : les cinq barres ne représentent pas cinq véhicules présents dans la rue.


Toutes les barres viennent du calcul réalisé sur cette photo : aucun score n'est écrit à l'avance.
Des catégories proches peuvent se partager les scores. Certaines dénominations anglaises
peuvent être plus spécialisées que le mot que nous aurions choisi.
`minibus` signifie petit bus, `police_van` fourgon de police, et `trolleybus` bus électrique
alimenté par des fils aériens. Si ces noms apparaissent, comparer ce qu'ils décrivent
à la photo : une ressemblance visuelle ne confirme pas le type exact du véhicule.

**Un score élevé n'est pas une fiabilité mesurée.** Par exemple, afficher 90 % ne prouve pas
que le modèle a raison sur 90 photos sur 100 de notre usage. Pour vérifier cela, il faudrait
un ensemble d'images représentatives dont les bonnes réponses sont connues.

De plus, ce modèle propose toujours des scores parmi ses catégories : une image inhabituelle
ne reçoit pas automatiquement une catégorie « inconnu ».


### À essayer — quelle part reste en dehors du top-5 ?

Compléter `part_restante` : à partir des cinq scores affichés, renvoyer **1 moins leur somme**.
`sum(scores)` additionne les nombres d'une liste. Ne pas arrondir le résultat dans la fonction.

Ce calcul concerne les autres catégories connues du modèle, pas une catégorie « inconnu ».


In [ ]:
def part_restante(scores):
    # À compléter : additionner les scores, puis retirer cette somme de 1.
    return None

reste = part_restante(scores_top5)
if reste is None:
    print("Compléter part_restante pour obtenir la part des autres catégories.")
else:
    print(f"Part des catégories absentes du top-5 : {reste:.1%}")


In [ ]:
# Les nombres ci-dessous sont fictifs : ils servent seulement à vérifier la fonction.
if part_restante([0.5, 0.2, 0.1, 0.05, 0.05]) is not None:
    # abs mesure l'écart ; la tolérance accepte les petites imprécisions des décimaux.
    assert abs(part_restante([0.5, 0.2, 0.1, 0.05, 0.05]) - 0.1) < 1e-9
    assert abs(part_restante([0.8, 0.05, 0.05, 0.05, 0.05])) < 1e-9
    assert abs(part_restante([0.2, 0.1, 0.1, 0.1, 0.1]) - 0.4) < 1e-9
    print("Les trois tests passent.")
else:
    print("Tests en attente : la fonction est encore vide.")


Si les cinq premiers scores totalisent 0,80, quelle part reste-t-il ?
Cela signifie-t-il « 20 % de chance que l'image contienne un objet inconnu » ?

Votre explication : …


## 8. Séparer le calcul du modèle et notre décision

Le modèle fournit un score. **L'application choisit quoi en faire.**
Un **seuil** est une limite numérique utilisée pour prendre une décision.
Nous allons proposer l'étiquette si son score atteint 0,80 ; sinon, demander une vérification.

![Un seuil sépare une étiquette proposée d’une réponse à vérifier ; il ne prouve pas la justesse du résultat.](../ressources/illustrations/02_score_decision.svg)

**0,80 est un choix d'exercice, pas une valeur validée pour un usage réel.**
Le relever produit davantage de demandes de vérification ; l'abaisser laisse passer plus de propositions.
Pour choisir un seuil utile, il faudrait mesurer les erreurs sur des images représentatives.


### À essayer — proposer ou demander une vérification

Compléter `decider` pour renvoyer :

- `"proposer"` si le score est **supérieur ou égal** au seuil ;
- `"à vérifier"` sinon.

Les vérifications des valeurs sont déjà écrites : un score et un seuil doivent être compris
entre 0 et 1. `raise ValueError(...)` signale une valeur non acceptable.
Le mot `return` renvoie la réponse de la fonction.


In [ ]:
def decider(score, seuil):
    if not 0 <= score <= 1:
        raise ValueError("Le score doit être compris entre 0 et 1.")
    if not 0 <= seuil <= 1:
        raise ValueError("Le seuil doit être compris entre 0 et 1.")
    # À compléter : comparer le score au seuil et renvoyer le bon texte.
    return None

decision = decider(score, seuil=0.80)
if decision is None:
    print("Compléter decider pour appliquer notre règle.")
else:
    print(f"{etiquette} — score {score:.1%} — décision : {decision}")


Les nombres des tests suivants sont **inventés** : ils vérifient notre règle
Python, pas la qualité du modèle. Le cas `score == seuil` vérifie que notre
choix est bien « supérieur **ou égal** », contrairement au seuillage strict
des pixels dans le notebook 01.


In [ ]:
if decider(0.80, 0.80) is not None:
    assert decider(0.82, 0.80) == "proposer"
    assert decider(0.80, 0.80) == "proposer"  # Tester la frontière : l'égalité compte.
    assert decider(0.79, 0.80) == "à vérifier"
    assert decider(0.0, 0.0) == "proposer"
    assert decider(1.0, 1.0) == "proposer"
    # Une valeur hors de [0, 1] doit provoquer l'erreur prévue par la fonction.
    try:
        decider(0.82, 1.20)
    except ValueError:
        print("Un seuil invalide est bien refusé.")
    else:
        raise AssertionError("Un seuil supérieur à 1 doit être refusé.")
    # Vérifier aussi un score invalide, pas seulement un seuil invalide.
    try:
        decider(-0.1, 0.50)
    except ValueError:
        print("Un score invalide est bien refusé.")
    else:
        raise AssertionError("Un score négatif doit être refusé.")
    print("Les tests de décision passent.")
else:
    print("Tests en attente : la fonction est encore vide.")


Avec un seuil de 0,80, que devient une proposition dont le score vaut exactement 0,80 ?
Et un score de 0,95 garantit-il une réponse correcte ?

Votre explication : …


## 9. Que se passe-t-il si l'on ne montre qu'un morceau ?

Reprenons le recadrage découvert avec OpenCV. Cette fois, nous allons volontairement conserver
la partie basse de la photo : **le modèle reçoit une autre image**, avec une autre répartition
des objets et du décor.

`image_bgr[y1:y2, x1:x2]` sélectionne des lignes puis des colonnes.
`hauteur // 2` calcule le milieu vertical avec une division entière.
Nous comparerons les deux propositions : elles peuvent changer, mais ce n'est pas garanti.
Ce recadrage volontaire n'est pas la reconstitution du prétraitement interne de `predict`.


In [ ]:
hauteur, largeur = image_bgr.shape[:2]  # Les deux premières dimensions de la photo.
morceau_bgr = image_bgr[hauteur // 2:hauteur, 0:largeur].copy()  # Garder le bas.

resultat_morceau = classifieur.predict(
    source=morceau_bgr,
    imgsz=224,
    device="cpu",
    verbose=False,
    save=False,
)[0]
numero_morceau = resultat_morceau.probs.top1
etiquette_morceau = resultat_morceau.names[numero_morceau]
score_morceau = float(resultat_morceau.probs.top1conf)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image_rgb)
axes[0].set_title(f"Photo complète\n{etiquette} — {score:.1%}")

# Le morceau est encore en BGR : le convertir seulement pour l'afficher.
axes[1].imshow(cv2.cvtColor(morceau_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Partie basse seulement\n{etiquette_morceau} — {score_morceau:.1%}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


**Figure de référence · visible sans exécution**

![Classification de la photo entière comparée à celle de sa partie basse.](../ressources/illustrations/02_effet_recadrage.png)

Le modèle reste identique ; seuls les pixels fournis changent. Dans cet essai, la première proposition passe de `minibus` à `umbrella` (parapluie), avec un score faible. Une étiquette en tête ne suffit donc pas à prouver sa justesse.


Comparer les deux étiquettes **et** les deux scores. Une étiquette identique ne signifie pas
que le modèle a vu la même chose ; un score plus élevé ne prouve pas que le recadrage l'a amélioré.
Nous avons changé les pixels et la question implicite sur le contenu de l'image.
Une proposition étonnante, comme `umbrella` (« parapluie »), reste possible :
le modèle classe l'image parmi ses catégories connues, même si aucune ne convient bien.

Surtout, aucun de ces résultats n'indique les coordonnées du bus ou des personnes.
**Classer deux images ne revient pas à détecter les objets de la photo d'origine.**


## 10. Ce que notre chaîne sait faire — et ce qu'il lui manque

Nous avons relié **photo → préparation → modèle local → scores → proposition → décision**.
Le modèle préentraîné évite un entraînement, mais pas le travail d'interprétation et de vérification.

| Notre question | Ce que nous avons obtenu |
|---|---|
| Quelle catégorie proposer pour l'image ? | Une étiquette globale et les autres scores. |
| Faut-il faire vérifier cette proposition ? | Une règle explicite, dont le seuil reste à valider. |
| Quels objets sont présents et où ? | Pas encore de réponse : il faut de la détection. |

Le prochain modèle proposera des **rectangles autour des objets**. Son vocabulaire sera différent :
**80 catégories COCO**, contre les **1 000 catégories ImageNet** de ce classifieur.
Changer de fichier de modèle change donc la tâche et les réponses possibles, pas seulement la taille du fichier.

[← Préparer une image](01_comprendre_et_preparer_une_image.ipynb)
· [Suite : détecter avec YOLO →](03_detecter_avec_yolo.ipynb)
